# NumPy Basics for Protein AI

This notebook covers essential NumPy operations for protein data manipulation.

**Learning Objectives:**
- Create and manipulate NumPy arrays
- Understand broadcasting for efficient computations
- Compute distance matrices for protein structures
- Apply linear algebra operations for structure alignment

In [ ]:
# Setup - Run this cell first
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

print(f"NumPy version: {np.__version__}")

## 1. Array Creation and Properties

NumPy arrays are the fundamental data structure for representing:
- Protein sequences as numerical encodings
- 3D coordinates of atoms
- Feature matrices and embeddings

In [ ]:
# Creating arrays from lists
coords_list = [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [1.5, 1.0, 0.0]]
coords = np.array(coords_list)
print(f"Coordinates:\n{coords}")
print(f"Shape: {coords.shape}")
print(f"Data type: {coords.dtype}")

In [ ]:
# Common array creation functions
n_residues = 76  # Length of ubiquitin
n_features = 20  # 20 amino acids

# Zeros - for one-hot encoding
one_hot = np.zeros((n_residues, n_features))
print(f"One-hot shape: {one_hot.shape}")

# Random - simulated coordinates
random_coords = np.random.randn(n_residues, 3) * 10  # Mean 0, scaled
print(f"Random coords shape: {random_coords.shape}")

# Eye - identity matrix for rotations
identity = np.eye(3)
print(f"Identity matrix:\n{identity}")

## 2. Array Indexing and Slicing

Efficient indexing is crucial for selecting atoms, residues, or features.

In [ ]:
# Create sample backbone coordinates (N, CA, C, O for each residue)
n_residues = 10
backbone = np.random.randn(n_residues, 4, 3)  # (residues, atoms, xyz)
print(f"Backbone shape: {backbone.shape}")

# Extract all CA atoms (index 1)
ca_coords = backbone[:, 1, :]
print(f"CA coords shape: {ca_coords.shape}")

# First 5 residues
first_five = backbone[:5]
print(f"First 5 residues shape: {first_five.shape}")

# Every other residue
every_other = backbone[::2]
print(f"Every other residue shape: {every_other.shape}")

In [ ]:
# Boolean indexing - very useful for filtering
# Example: select residues with CA z-coordinate > 0
ca_z = backbone[:, 1, 2]  # z-coordinate of CA atoms
mask = ca_z > 0
print(f"Mask: {mask}")

selected_residues = backbone[mask]
print(f"Selected residues shape: {selected_residues.shape}")

## 3. Broadcasting

Broadcasting allows NumPy to perform operations on arrays with different shapes. This is essential for efficient protein computations.

In [ ]:
# Example 1: Centering coordinates
coords = np.random.randn(100, 3)  # 100 atoms
centroid = coords.mean(axis=0)    # Shape: (3,)

print(f"Coords shape: {coords.shape}")
print(f"Centroid shape: {centroid.shape}")

# Broadcasting: (100, 3) - (3,) -> (100, 3)
centered = coords - centroid
print(f"Centered coords shape: {centered.shape}")
print(f"New centroid (should be ~0): {centered.mean(axis=0)}")

In [ ]:
# Example 2: Scaling each feature differently
features = np.random.randn(100, 20)  # 100 samples, 20 features
scales = np.array([1, 2, 3, 4, 5] * 4)  # Different scale per feature

# Broadcasting: (100, 20) * (20,) -> (100, 20)
scaled = features * scales
print(f"Scaled features shape: {scaled.shape}")

In [ ]:
# Example 3: Pairwise differences (key for distance matrix)
n = 5
coords = np.random.randn(n, 3)

# Add dimensions for broadcasting
# (n, 1, 3) - (1, n, 3) -> (n, n, 3)
diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
print(f"Pairwise differences shape: {diff.shape}")

## 4. Distance Matrix Computation

The distance matrix is fundamental for protein structure analysis. It shows the pairwise distances between all atoms or residues.

In [ ]:
def compute_distance_matrix(coords):
    """
    Compute pairwise Euclidean distances.
    
    Args:
        coords: (N, 3) array of coordinates
    
    Returns:
        (N, N) distance matrix
    """
    # Pairwise differences using broadcasting
    diff = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    # Euclidean distance
    dist_matrix = np.sqrt(np.sum(diff ** 2, axis=-1))
    return dist_matrix

# Test with random coordinates
n_atoms = 50
coords = np.random.randn(n_atoms, 3) * 10
dist_matrix = compute_distance_matrix(coords)

print(f"Distance matrix shape: {dist_matrix.shape}")
print(f"Min distance: {dist_matrix[dist_matrix > 0].min():.2f}")
print(f"Max distance: {dist_matrix.max():.2f}")

In [ ]:
# Visualize distance matrix
plt.figure(figsize=(8, 6))
plt.imshow(dist_matrix, cmap='viridis')
plt.colorbar(label='Distance (Å)')
plt.xlabel('Atom Index')
plt.ylabel('Atom Index')
plt.title('Pairwise Distance Matrix')
plt.show()

In [ ]:
# Contact map from distance matrix
threshold = 8.0  # Angstroms - typical contact threshold
contact_map = dist_matrix < threshold

plt.figure(figsize=(8, 6))
plt.imshow(contact_map, cmap='Blues')
plt.xlabel('Residue Index')
plt.ylabel('Residue Index')
plt.title(f'Contact Map (threshold = {threshold}Å)')
plt.show()

# Count contacts (excluding diagonal)
n_contacts = (contact_map.sum() - n_atoms) // 2
print(f"Number of contacts: {n_contacts}")

## 5. Linear Algebra for Structure Alignment

Structure alignment is essential for comparing proteins. The Kabsch algorithm finds the optimal rotation to superimpose two structures.

In [ ]:
def compute_rmsd(coords1, coords2):
    """
    Compute Root Mean Square Deviation between two structures.
    
    Args:
        coords1, coords2: (N, 3) coordinate arrays
    
    Returns:
        RMSD value
    """
    diff = coords1 - coords2
    return np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))

# Create two structures
original = np.random.randn(50, 3)

# Apply known rotation and translation
angle = np.pi / 4  # 45 degrees
R_known = np.array([
    [np.cos(angle), -np.sin(angle), 0],
    [np.sin(angle),  np.cos(angle), 0],
    [0,              0,             1]
])
t_known = np.array([10, 5, 3])

transformed = original @ R_known.T + t_known

print(f"RMSD before alignment: {compute_rmsd(original, transformed):.2f}")

In [ ]:
def kabsch_align(mobile, target):
    """
    Align mobile structure to target using Kabsch algorithm.
    
    Args:
        mobile: (N, 3) coordinates to be aligned
        target: (N, 3) reference coordinates
    
    Returns:
        Aligned mobile coordinates
    """
    # Center both structures
    mobile_center = mobile.mean(axis=0)
    target_center = target.mean(axis=0)
    
    mobile_centered = mobile - mobile_center
    target_centered = target - target_center
    
    # Compute optimal rotation using SVD
    H = mobile_centered.T @ target_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    
    # Handle reflection case (det(R) should be +1)
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    
    # Apply rotation and translation
    aligned = mobile_centered @ R + target_center
    
    return aligned

# Align transformed back to original
aligned = kabsch_align(transformed, original)
print(f"RMSD after alignment: {compute_rmsd(original, aligned):.6f}")

## 6. Practical Exercise: Protein Analysis

Let's put it all together with a simulated protein structure.

In [ ]:
# Simulate a small protein (like crambin, 46 residues)
n_residues = 46

# Generate a coiled structure
t = np.linspace(0, 4*np.pi, n_residues)
x = 10 * np.cos(t)
y = 10 * np.sin(t)
z = 3.8 * np.arange(n_residues)  # 3.8Å per residue along axis

ca_coords = np.column_stack([x, y, z])
print(f"CA coordinates shape: {ca_coords.shape}")

In [ ]:
# Visualize structure in 2D projections
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XY projection
axes[0].plot(ca_coords[:, 0], ca_coords[:, 1], 'b-o', markersize=3)
axes[0].set_xlabel('X (Å)')
axes[0].set_ylabel('Y (Å)')
axes[0].set_title('XY Projection (Top View)')
axes[0].axis('equal')

# XZ projection
axes[1].plot(ca_coords[:, 0], ca_coords[:, 2], 'b-o', markersize=3)
axes[1].set_xlabel('X (Å)')
axes[1].set_ylabel('Z (Å)')
axes[1].set_title('XZ Projection (Side View)')

# YZ projection
axes[2].plot(ca_coords[:, 1], ca_coords[:, 2], 'b-o', markersize=3)
axes[2].set_xlabel('Y (Å)')
axes[2].set_ylabel('Z (Å)')
axes[2].set_title('YZ Projection (Side View)')

plt.tight_layout()
plt.show()

In [ ]:
# Compute and analyze distance matrix
dist_matrix = compute_distance_matrix(ca_coords)

# Create contact map
contact_map = dist_matrix < 8.0

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im1 = axes[0].imshow(dist_matrix, cmap='viridis')
axes[0].set_title('Distance Matrix')
axes[0].set_xlabel('Residue')
axes[0].set_ylabel('Residue')
plt.colorbar(im1, ax=axes[0], label='Distance (Å)')

im2 = axes[1].imshow(contact_map, cmap='Blues')
axes[1].set_title('Contact Map (8Å threshold)')
axes[1].set_xlabel('Residue')
axes[1].set_ylabel('Residue')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze contacts by sequence separation
separations = []
for i in range(n_residues):
    for j in range(i+1, n_residues):
        if contact_map[i, j]:
            separations.append(j - i)

plt.figure(figsize=(10, 5))
plt.hist(separations, bins=range(1, max(separations)+2), edgecolor='black', alpha=0.7)
plt.xlabel('Sequence Separation')
plt.ylabel('Number of Contacts')
plt.title('Contact Distribution by Sequence Separation')
plt.axvline(x=6, color='r', linestyle='--', label='Short-range cutoff')
plt.legend()
plt.show()

# Count local vs long-range contacts
local_contacts = sum(1 for s in separations if s < 6)
long_range = sum(1 for s in separations if s >= 6)
print(f"Local contacts (|i-j| < 6): {local_contacts}")
print(f"Long-range contacts (|i-j| >= 6): {long_range}")

## Summary

In this notebook, we covered:

1. **Array creation**: zeros, randn, eye, and array conversion
2. **Indexing**: slicing, boolean indexing for filtering
3. **Broadcasting**: efficient operations on different-shaped arrays
4. **Distance matrices**: fundamental for structure analysis
5. **Kabsch alignment**: optimal superposition using SVD

These operations form the foundation for all protein AI computations.